# LeetCode 641 — Design Circular Deque bằng Python

Link đề: https://leetcode.com/problems/design-circular-deque/ · Độ khó: **Medium** · Chủ đề: Design, Queue, Array, Linked List

Skeleton LeetCode:

```python
class MyCircularDeque:

    def __init__(self, k: int):
        ...

    def insertFront(self, value: int) -> bool:
        ...

    def insertLast(self, value: int) -> bool:
        ...

    def deleteFront(self) -> bool:
        ...

    def deleteLast(self) -> bool:
        ...

    def getFront(self) -> int:
        ...

    def getRear(self) -> int:
        ...

    def isEmpty(self) -> bool:
        ...

    def isFull(self) -> bool:
        ...


# Your MyCircularDeque object will be instantiated and called as such:
# obj = MyCircularDeque(k)
# param_1 = obj.insertFront(value)
# param_2 = obj.insertLast(value)
# param_3 = obj.deleteFront()
# param_4 = obj.deleteLast()
# param_5 = obj.getFront()
# param_6 = obj.getRear()
# param_7 = obj.isEmpty()
# param_8 = obj.isFull()
```

Ghi chú: đây là bài **thiết kế lớp**. LeetCode cho hai danh sách: tên lệnh và tham số, rồi tự gọi từng lệnh lên đối tượng. Notebook tự dựng hàm `run_ops(cls, names, args)` mô phỏng đúng cách chạy đó. Các ví dụ thêm ngoài ví dụ của đề (mục 2) do tôi tự đặt, đáp án đã kiểm chứng bằng code; nếu cần khớp từng chữ, hãy đối chiếu lại đề gốc.

## 1. Mục tiêu học

Sau notebook này bạn sẽ:

1. Hiểu đề: thiết kế một **hàng đợi hai đầu (deque)** có **sức chứa tối đa `k`**, thêm/xóa/xem được ở **cả đầu và cuối**, mọi thao tác chạy `O(1)`, và biết khi nào đầy (`isFull`) hay rỗng (`isEmpty`).
2. Nắm các công cụ Python: viết `class` với `__init__` và `self`, tạo list `[0] * k`, phép chia dư `%` với số **âm** (`-1 % 3 == 2`), khái niệm **ring buffer (bộ đệm vòng)**, nút liên kết đôi `Node` có `prev`/`next`, nút **sentinel (lính canh)**, và `collections.deque`.
3. Đi từ **list thường dùng `insert(0, x)`/`pop(0)` (O(k) ở đầu)** → **bộ đệm vòng với `front` và `size`** → **bộ đệm vòng `k + 1` ô với `front`/`rear`** → **danh sách liên kết đôi có sentinel** → **bọc `collections.deque` có kiểm tra sức chứa**.
4. Hiểu mấu chốt: đánh số ô **theo vòng tròn bằng phép `% k`**; đi lùi ở đầu (`front - 1`) và đi tiến ở cuối (`front + size`); và phải có cách **phân biệt "rỗng" với "đầy"** (dùng biến `size`, hoặc chừa một ô trống).
5. Tránh các lỗi kinh điển: `front == rear` vừa là rỗng vừa là đầy, `getRear` lệch một ô, quên dời `front` khi `deleteFront`, trả giá trị rác thay vì `-1` khi rỗng, và quên `% k` khiến chỉ số trôi ra ngoài mảng.

## 2. Đề bài & ví dụ

Thiết kế lớp **`MyCircularDeque`** (hàng đợi hai đầu dạng vòng) với các thao tác:

- `MyCircularDeque(k)`: khởi tạo deque với sức chứa tối đa **`k`** phần tử.
- `insertFront(value)`: thêm `value` vào **đầu**. Trả `True` nếu thành công, `False` nếu deque đã đầy.
- `insertLast(value)`: thêm `value` vào **cuối**. Trả `True` nếu thành công, `False` nếu đã đầy.
- `deleteFront()`: xóa phần tử ở **đầu**. Trả `True` nếu thành công, `False` nếu rỗng.
- `deleteLast()`: xóa phần tử ở **cuối**. Trả `True` nếu thành công, `False` nếu rỗng.
- `getFront()`: trả phần tử ở đầu, hoặc `-1` nếu rỗng.
- `getRear()`: trả phần tử ở cuối, hoặc `-1` nếu rỗng.
- `isEmpty()`: `True` nếu rỗng.
- `isFull()`: `True` nếu đầy.

**Ví dụ 1** (của đề)

- Lệnh: `["MyCircularDeque","insertLast","insertLast","insertFront","insertFront","getRear","isFull","deleteLast","insertFront","getFront"]`
- Tham số: `[[3],[1],[2],[3],[4],[],[],[],[4],[]]`
- Kết quả: `[null,true,true,true,false,2,true,true,true,4]`
- Giải thích: `k = 3`. Thêm cuối `1`, thêm cuối `2` → `[1,2]`; thêm đầu `3` → `[3,1,2]` (đầy); thêm đầu `4` thất bại (`false`); `getRear` là `2`; `isFull` là `true`; xóa cuối → `[3,1]`; thêm đầu `4` → `[4,3,1]`; `getFront` là `4`.

**Ví dụ 2 (tự đặt, đối chiếu lại đề gốc nếu cần)**

- Lệnh: `["MyCircularDeque","isEmpty","getFront","getRear","deleteFront","deleteLast","insertLast","getFront","getRear","isFull"]`, tham số: `[[1],[],[],[],[],[],[7],[],[],[]]`
- Kết quả: `[null,true,-1,-1,false,false,true,7,7,true]`
- Giải thích: deque sức chứa `1`: lúc đầu rỗng nên mọi lệnh xóa/xem đều thất bại hoặc trả `-1`; sau khi chèn `7` thì cả đầu lẫn cuối đều là `7` và deque đã đầy.

**Ràng buộc:**

- `1 <= k <= 1000`
- `0 <= value <= 1000`
- Có tối đa `2000` lời gọi tới `insertFront`, `insertLast`, `deleteFront`, `deleteLast`, `getFront`, `getRear`, `isEmpty`, `isFull`.

Chú ý: với tối đa `2000` lời gọi thì cả cách `O(k)` mỗi lệnh vẫn qua được; nhưng mục tiêu của bài là học thiết kế mỗi lệnh `O(1)`.

In [1]:
import random
import time
from collections import deque
from typing import List, Optional


def run_ops(cls, names, args):
    """Mô phỏng cách LeetCode chạy bài thiết kế lớp.
    names[0] là tên lớp (hàm dựng), các tên sau là tên phương thức; args là list tham số tương ứng.
    Trả về list kết quả, với hàm dựng cho None (in ra là null)."""
    obj, out = None, []
    for name, a in zip(names, args):
        if obj is None:
            obj = cls(*a)
            out.append(None)
        else:
            out.append(getattr(obj, name)(*a))
    return out


def show(res):
    """In kết quả theo kiểu LeetCode: None -> null, True -> true."""
    def f(x):
        return "null" if x is None else str(x).lower() if isinstance(x, bool) else str(x)
    return "[" + ",".join(f(x) for x in res) + "]"


ex1 = (
    ["MyCircularDeque", "insertLast", "insertLast", "insertFront", "insertFront", "getRear", "isFull", "deleteLast", "insertFront", "getFront"],
    [[3], [1], [2], [3], [4], [], [], [], [4], []],
    [None, True, True, True, False, 2, True, True, True, 4],
)
ex2 = (
    ["MyCircularDeque", "isEmpty", "getFront", "getRear", "deleteFront", "deleteLast", "insertLast", "getFront", "getRear", "isFull"],
    [[1], [], [], [], [], [], [7], [], [], []],
    [None, True, -1, -1, False, False, True, 7, 7, True],
)
examples = [ex1, ex2]
for names, args, expected in examples:
    print(names[:4], "...", "-> kỳ vọng", show(expected))

['MyCircularDeque', 'insertLast', 'insertLast', 'insertFront'] ... -> kỳ vọng [null,true,true,true,false,2,true,true,true,4]
['MyCircularDeque', 'isEmpty', 'getFront', 'getRear'] ... -> kỳ vọng [null,true,-1,-1,false,false,true,7,7,true]


## 3. Kiến thức nền

### 3.1. Lớp, `__init__`, `self` và cách LeetCode gọi

Một `class` gom **dữ liệu** (thuộc tính `self.x`) và **hành động** (phương thức). `__init__(self, k)` chạy khi tạo đối tượng `MyCircularDeque(3)`; `self` là chính đối tượng đang được tạo/gọi. Mỗi lần gọi `obj.insertLast(5)` Python tự truyền `obj` vào tham số `self`.

Bài thiết kế lớp **không có hàm `main`**: LeetCode đọc hai danh sách (tên lệnh, tham số), tạo đối tượng bằng lệnh đầu rồi gọi tuần tự các lệnh còn lại bằng `getattr(obj, name)(*args)`. Hàm `run_ops` ở cell trên làm đúng việc đó.

In [2]:
class Counter:
    def __init__(self, start):
        self.value = start            # thuộc tính của đối tượng

    def bump(self):
        self.value += 1
        return self.value


c = Counter(10)
print(c.bump(), c.bump())                      # 11 12
print(getattr(c, "bump")())                    # 13: gọi phương thức bằng tên chuỗi
print(run_ops(Counter, ["Counter", "bump", "bump"], [[5], [], []]))   # [None, 6, 7]

11 12
13
[None, 6, 7]


### 3.2. Phép `%` và chỉ số vòng tròn

Muốn đi vòng quanh `k` ô đánh số `0..k-1`, ta dùng `% k`: sau ô `k - 1` là ô `0`. Quan trọng: trong Python, `%` với số **âm** vẫn cho kết quả **không âm** khi chia cho số dương (`-1 % 3 == 2`), nên đi **lùi** một ô là `(i - 1) % k`. (Ở C/Java `-1 % 3` bằng `-1`, người ta phải viết `(i - 1 + k) % k`; viết kiểu này cũng đúng trong Python.)

In [3]:
k = 5
print([(i + 1) % k for i in range(k)])          # đi tiến:  1 2 3 4 0
print([(i - 1) % k for i in range(k)])          # đi lùi :  4 0 1 2 3
print([(i - 1 + k) % k for i in range(k)])      # cách viết "an toàn" kiểu C/Java: cùng kết quả
print(-1 % 3, -7 % 3, 7 % 3)                    # 2 2 1

[1, 2, 3, 4, 0]
[4, 0, 1, 2, 3]
[4, 0, 1, 2, 3]
2 2 1


### 3.3. Bộ đệm vòng (ring buffer)

Ta cấp phát **một mảng cố định `q` dài `k`** và coi nó như một **vòng tròn**. Deque là một "đoạn liên tiếp trên vòng" bắt đầu tại ô `front` và dài `size` ô:

```
chỉ số:   0    1    2    3    4        k = 5, front = 3, size = 4
mảng q:  [ c ][ d ][ . ][ a ][ b ]      nội dung deque (đầu -> cuối): a b c d
                       ^front           ô cuối là (front + size - 1) % 5 = 1
```

Từ hai số `front` và `size` ta suy ra mọi thứ:

- ô đầu: `front`; ô cuối: `(front + size - 1) % k`; ô trống kế cuối: `(front + size) % k`; ô trống kế đầu: `(front - 1) % k`.
- rỗng: `size == 0`; đầy: `size == k`.

Không cần di chuyển phần tử nào: thêm/xóa chỉ **dời `front` và đổi `size`**, nên mỗi thao tác `O(1)`.

In [4]:
def logical_view(q, front, size):
    """Đọc nội dung deque (đầu -> cuối) từ mảng vòng."""
    cap = len(q)
    return [q[(front + i) % cap] for i in range(size)]


q = ["c", "d", ".", "a", "b"]
print(logical_view(q, front=3, size=4))          # ['a', 'b', 'c', 'd']
print("ô cuối:", (3 + 4 - 1) % 5, "| ô trống kế cuối:", (3 + 4) % 5, "| ô trống kế đầu:", (3 - 1) % 5)

['a', 'b', 'c', 'd']
ô cuối: 1 | ô trống kế cuối: 2 | ô trống kế đầu: 2


### 3.4. Vấn đề "rỗng hay đầy?" khi chỉ có `front` và `rear`

Nếu chỉ lưu hai con trỏ `front` (ô đầu) và `rear` (ô trống kế cuối) trên mảng `k` ô, thì cả "rỗng" lẫn "đầy" đều cho `front == rear` (đã đi đúng một vòng) — **không phân biệt được**. Có hai cách thoát:

- **Thêm biến `size`** (cách 2): rỗng là `size == 0`, đầy là `size == k`.
- **Chừa một ô trống**: dùng mảng `k + 1` ô; **rỗng** khi `front == rear`, **đầy** khi `(rear + 1) % (k + 1) == front` (cách 3).

### 3.5. Danh sách liên kết đôi và nút sentinel

Mỗi **nút** có `val`, `prev` (nút trước), `next` (nút sau). Deque là một dây nút nối hai chiều. **Sentinel** (lính canh) là hai nút giả `head` và `tail` luôn có mặt ở hai đầu, không chứa dữ liệu thật. Nhờ chúng, thêm/xóa ở đầu/cuối **luôn** có nút ở hai phía, không cần xử lý riêng "danh sách rỗng" hay "xóa nút duy nhất".

```
head <-> [a] <-> [b] <-> [c] <-> tail
```

In [5]:
class Node:
    def __init__(self, val=0):
        self.val = val
        self.prev = None
        self.next = None


head, tail = Node(), Node()
head.next, tail.prev = tail, head            # ban đầu: head <-> tail (rỗng)


def link_after(node, new):
    """Chèn nút new ngay sau node."""
    new.prev, new.next = node, node.next
    node.next.prev = new
    node.next = new


for v in (1, 2, 3):
    link_after(tail.prev, Node(v))           # chèn trước tail (tức thêm cuối)

cur, vals = head.next, []
while cur is not tail:
    vals.append(cur.val)
    cur = cur.next
print(vals)                                   # [1, 2, 3]

[1, 2, 3]


### 3.6. `collections.deque`

Python có sẵn `collections.deque`: `appendleft`, `append`, `popleft`, `pop`, truy cập `d[0]`, `d[-1]`, `len(d)`; các thao tác ở hai đầu đều `O(1)`. Nó **không giới hạn sức chứa** (trừ khi truyền `maxlen`, nhưng khi đầy nó **tự đẩy** phần tử đầu kia ra thay vì báo lỗi), vì vậy ta phải **tự kiểm tra `len(d) == k`** trước khi chèn. Trong cuộc phỏng vấn, người ra đề thường muốn bạn **tự cài**, không dùng sẵn.

In [6]:
d = deque()
d.append(1); d.append(2); d.appendleft(3)
print(d, d[0], d[-1], len(d))                 # deque([3, 1, 2]) 3 2 3
print(d.popleft(), d.pop(), d)                # 3 2 deque([1])

m = deque(maxlen=2)
m.append(1); m.append(2); m.append(3)         # tự đẩy 1 ra
print(m)                                      # deque([2, 3], maxlen=2)

deque([3, 1, 2]) 3 2 3
3 2 deque([1])
deque([2, 3], maxlen=2)


## 4. Ý tưởng nền tảng / cách nghĩ trực giác (tính tay)

Hãy vẽ ví dụ 1 trên mảng `q` 3 ô, `front = 0`, `size = 0`:

| lệnh | tính | `q` | `front` | `size` | nội dung (đầu→cuối) |
|---|---|---|---|---|---|
| `insertLast(1)` | ô `(0+0) % 3 = 0` | `[1,_,_]` | 0 | 1 | `1` |
| `insertLast(2)` | ô `(0+1) % 3 = 1` | `[1,2,_]` | 0 | 2 | `1 2` |
| `insertFront(3)` | `front = (0-1) % 3 = 2`, ghi ô 2 | `[1,2,3]` | 2 | 3 | `3 1 2` |
| `insertFront(4)` | đầy (`size == 3`) → `false` | `[1,2,3]` | 2 | 3 | `3 1 2` |
| `getRear()` | ô `(2+3-1) % 3 = 1` | | 2 | 3 | trả `2` |
| `deleteLast()` | `size = 2` | `[1,2,3]` | 2 | 2 | `3 1` |
| `insertFront(4)` | `front = (2-1) % 3 = 1`, ghi ô 1 | `[1,4,3]` | 1 | 3 | `4 3 1` |
| `getFront()` | ô `front = 1` | | 1 | 3 | trả `4` |

Nhận xét: chèn đầu là "lùi `front` một ô rồi ghi", chèn cuối là "ghi vào ô `front + size`", xóa đầu là "tiến `front` một ô", xóa cuối chỉ là "giảm `size`". Mảng không bao giờ phải dịch chuyển.

Cell dưới kiểm chứng tính tay đó bằng code (ta sẽ viết lớp thật ở các mục sau).

In [7]:
# kiểm chứng nhanh bảng tính tay bằng đúng các phép tính ở trên
q, front, size, cap = [None] * 3, 0, 0, 3
q[(front + size) % cap] = 1; size += 1                   # insertLast(1)
q[(front + size) % cap] = 2; size += 1                   # insertLast(2)
front = (front - 1) % cap; q[front] = 3; size += 1       # insertFront(3)
print(q, front, size, logical_view(q, front, size))      # [1, 2, 3] 2 3 [3, 1, 2]
print("getRear:", q[(front + size - 1) % cap])           # 2
size -= 1                                                # deleteLast
front = (front - 1) % cap; q[front] = 4; size += 1       # insertFront(4)
print(q, front, size, logical_view(q, front, size))      # [1, 4, 3] 1 3 [4, 3, 1]
print("getFront:", q[front])                             # 4

[1, 2, 3] 2 3 [3, 1, 2]
getRear: 2
[1, 4, 3] 1 3 [4, 3, 1]
getFront: 4


## 5. Cách 1 — Brute Force / baseline: dùng `list` thường, `insert(0, x)` và `pop(0)`

**Ý tưởng.** Dùng một `list` Python `self.q` để chứa deque đúng theo thứ tự đầu→cuối; `self.cap = k` để kiểm tra đầy.

- `insertFront`: nếu đầy thì `False`; ngược lại `q.insert(0, v)`.
- `insertLast`: `q.append(v)`.
- `deleteFront`: `q.pop(0)`; `deleteLast`: `q.pop()`.
- `getFront`: `q[0]`; `getRear`: `q[-1]`; rỗng thì `-1`.

Đơn giản, dễ đúng — rất hợp làm **mô hình tham chiếu (oracle)** khi test. Nhưng `insert(0, x)` và `pop(0)` phải **dịch chuyển mọi phần tử còn lại** nên chậm.

In [8]:
class DequeList:
    def __init__(self, k: int):
        self.q = []
        self.cap = k

    def insertFront(self, value: int) -> bool:
        if self.isFull():
            return False
        self.q.insert(0, value)          # dịch mọi phần tử sang phải: O(size)
        return True

    def insertLast(self, value: int) -> bool:
        if self.isFull():
            return False
        self.q.append(value)
        return True

    def deleteFront(self) -> bool:
        if self.isEmpty():
            return False
        self.q.pop(0)                    # dịch mọi phần tử sang trái: O(size)
        return True

    def deleteLast(self) -> bool:
        if self.isEmpty():
            return False
        self.q.pop()
        return True

    def getFront(self) -> int:
        return -1 if self.isEmpty() else self.q[0]

    def getRear(self) -> int:
        return -1 if self.isEmpty() else self.q[-1]

    def isEmpty(self) -> bool:
        return len(self.q) == 0

    def isFull(self) -> bool:
        return len(self.q) == self.cap


for names, args, expected in examples:
    got = run_ops(DequeList, names, args)
    print(show(got))
    assert got == expected

[null,true,true,true,false,2,true,true,true,4]
[null,true,-1,-1,false,false,true,7,7,true]


**Độ phức tạp.** `insertFront`/`deleteFront` là **`O(size)` = `O(k)`**, các lệnh còn lại `O(1)`; bộ nhớ `O(k)`. Với `k ≤ 1000` và `2000` lệnh thì vẫn qua, nhưng nếu `k` lên tới hàng triệu thì thảm họa.

**Nút thắt.** Ta **di chuyển phần tử** chỉ để chừa chỗ ở đầu list. Nếu thay vì dịch phần tử, ta **dịch "vị trí bắt đầu"** thì không cần di chuyển gì — đó là ý tưởng bộ đệm vòng.

## 6. Ý tưởng thuật toán / bước đột phá

Hai ý tưởng ghép lại:

1. **Đừng dịch dữ liệu, hãy dịch con trỏ.** Giữ mảng cố định `k` ô và một chỉ số `front`. Thêm vào đầu = lùi `front` một ô rồi ghi; xóa đầu = tiến `front` một ô.
2. **Vòng tròn bằng `% k`.** Mảng "dính" đầu với cuối nên con trỏ không bao giờ tràn: `% k` quay về ô đầu.
3. **Phân biệt rỗng/đầy** bằng thêm `size` (hoặc chừa một ô).

Nếu bạn không thích mảng, **danh sách liên kết đôi** cũng cho `O(1)` ở hai đầu vì có con trỏ tới cả hai đầu (nhờ sentinel). Và Python có sẵn `deque` cho các thao tác này.

## 7. Các cách tiếp theo

Tổng cộng 5 cách: cách 1 (list, mục 5) và bốn cách dưới đây.

### 7.1. Cách 2 — Bộ đệm vòng với `front` và `size`

**Giải thích từng đoạn.** Thuộc tính: `q = [0] * k`, `front = 0`, `size = 0`, `cap = k`.

- `insertFront(v)`: nếu đầy thì `False`. Lùi `front = (front - 1) % cap`, ghi `q[front] = v`, `size += 1`. (Khi rỗng, lùi `front` rồi ghi vẫn đúng: phần tử duy nhất nằm ở `front`.)
- `insertLast(v)`: nếu đầy thì `False`. Ô ghi là `(front + size) % cap`; `size += 1`.
- `deleteFront()`: nếu rỗng thì `False`. Tiến `front = (front + 1) % cap`, `size -= 1`.
- `deleteLast()`: nếu rỗng thì `False`. Chỉ cần `size -= 1`.
- `getFront()`: `q[front]`. `getRear()`: `q[(front + size - 1) % cap]`. Rỗng thì `-1`.

Hàm `view_ring` dưới đây chỉ dùng để **quan sát** trạng thái khi demo.

In [9]:
class DequeRingSize:
    def __init__(self, k: int):
        self.q = [0] * k
        self.front = 0
        self.size = 0
        self.cap = k

    def insertFront(self, value: int) -> bool:
        if self.isFull():
            return False
        self.front = (self.front - 1) % self.cap
        self.q[self.front] = value
        self.size += 1
        return True

    def insertLast(self, value: int) -> bool:
        if self.isFull():
            return False
        self.q[(self.front + self.size) % self.cap] = value
        self.size += 1
        return True

    def deleteFront(self) -> bool:
        if self.isEmpty():
            return False
        self.front = (self.front + 1) % self.cap
        self.size -= 1
        return True

    def deleteLast(self) -> bool:
        if self.isEmpty():
            return False
        self.size -= 1
        return True

    def getFront(self) -> int:
        return -1 if self.isEmpty() else self.q[self.front]

    def getRear(self) -> int:
        return -1 if self.isEmpty() else self.q[(self.front + self.size - 1) % self.cap]

    def isEmpty(self) -> bool:
        return self.size == 0

    def isFull(self) -> bool:
        return self.size == self.cap


def view_ring(d):
    return f"q={d.q} front={d.front} size={d.size} nội dung={logical_view(d.q, d.front, d.size)}"


d = DequeRingSize(3)
for name, a in zip(ex1[0][1:], ex1[1][1:]):
    r = getattr(d, name)(*a)
    print(f"{name:>12}{tuple(a)!s:>6} -> {str(r):<5} | {view_ring(d)}")

for names, args, expected in examples:
    assert run_ops(DequeRingSize, names, args) == expected
print("OK")

  insertLast  (1,) -> True  | q=[1, 0, 0] front=0 size=1 nội dung=[1]
  insertLast  (2,) -> True  | q=[1, 2, 0] front=0 size=2 nội dung=[1, 2]
 insertFront  (3,) -> True  | q=[1, 2, 3] front=2 size=3 nội dung=[3, 1, 2]
 insertFront  (4,) -> False | q=[1, 2, 3] front=2 size=3 nội dung=[3, 1, 2]
     getRear    () -> 2     | q=[1, 2, 3] front=2 size=3 nội dung=[3, 1, 2]
      isFull    () -> True  | q=[1, 2, 3] front=2 size=3 nội dung=[3, 1, 2]
  deleteLast    () -> True  | q=[1, 2, 3] front=2 size=2 nội dung=[3, 1]
 insertFront  (4,) -> True  | q=[1, 4, 3] front=1 size=3 nội dung=[4, 3, 1]
    getFront    () -> 4     | q=[1, 4, 3] front=1 size=3 nội dung=[4, 3, 1]
OK


**Code nộp LeetCode (cách 2)** — đây là bản ta sẽ nộp ở mục 9; các lớp `Deque2/3/4/5` trong bộ test là các cách đổi tên để so sánh. Để tránh lặp, ở đây ta chạy lại lớp `DequeRingSize` ở trên (nó **chính là** code nộp, chỉ khác tên lớp).

**Độ phức tạp cách 2.** Mọi thao tác **O(1)** thời gian; bộ nhớ **O(k)**.

### 7.2. Cách 3 — Bộ đệm vòng `k + 1` ô, dùng `front` và `rear` (không có `size`)

**Ý tưởng.** Dùng mảng **`k + 1` ô** (`n = k + 1`), lưu `front` (ô đầu) và `rear` (ô **trống** ngay sau phần tử cuối). Tối đa `k` phần tử, luôn chừa **1 ô trống** để phân biệt:

- rỗng: `front == rear`;
- đầy: `(rear + 1) % n == front`.

Thao tác:

- `insertFront(v)`: `front = (front - 1) % n; q[front] = v`.
- `insertLast(v)`: `q[rear] = v; rear = (rear + 1) % n`.
- `deleteFront()`: `front = (front + 1) % n`. `deleteLast()`: `rear = (rear - 1) % n`.
- `getFront()`: `q[front]`. `getRear()`: `q[(rear - 1) % n]`.

In [10]:
class DequeRingK1:
    def __init__(self, k: int):
        self.n = k + 1                     # k phần tử + 1 ô trống để phân biệt rỗng/đầy
        self.q = [0] * self.n
        self.front = 0
        self.rear = 0                      # ô trống kế phần tử cuối

    def insertFront(self, value: int) -> bool:
        if self.isFull():
            return False
        self.front = (self.front - 1) % self.n
        self.q[self.front] = value
        return True

    def insertLast(self, value: int) -> bool:
        if self.isFull():
            return False
        self.q[self.rear] = value
        self.rear = (self.rear + 1) % self.n
        return True

    def deleteFront(self) -> bool:
        if self.isEmpty():
            return False
        self.front = (self.front + 1) % self.n
        return True

    def deleteLast(self) -> bool:
        if self.isEmpty():
            return False
        self.rear = (self.rear - 1) % self.n
        return True

    def getFront(self) -> int:
        return -1 if self.isEmpty() else self.q[self.front]

    def getRear(self) -> int:
        return -1 if self.isEmpty() else self.q[(self.rear - 1) % self.n]

    def isEmpty(self) -> bool:
        return self.front == self.rear

    def isFull(self) -> bool:
        return (self.rear + 1) % self.n == self.front


d = DequeRingK1(3)
for name, a in zip(ex1[0][1:], ex1[1][1:]):
    r = getattr(d, name)(*a)
    print(f"{name:>12}{tuple(a)!s:>6} -> {str(r):<5} | q={d.q} front={d.front} rear={d.rear}")

for names, args, expected in examples:
    assert run_ops(DequeRingK1, names, args) == expected
print("OK")

  insertLast  (1,) -> True  | q=[1, 0, 0, 0] front=0 rear=1
  insertLast  (2,) -> True  | q=[1, 2, 0, 0] front=0 rear=2
 insertFront  (3,) -> True  | q=[1, 2, 0, 3] front=3 rear=2
 insertFront  (4,) -> False | q=[1, 2, 0, 3] front=3 rear=2
     getRear    () -> 2     | q=[1, 2, 0, 3] front=3 rear=2
      isFull    () -> True  | q=[1, 2, 0, 3] front=3 rear=2
  deleteLast    () -> True  | q=[1, 2, 0, 3] front=3 rear=1
 insertFront  (4,) -> True  | q=[1, 2, 4, 3] front=2 rear=1
    getFront    () -> 4     | q=[1, 2, 4, 3] front=2 rear=1
OK


**Độ phức tạp cách 3.** Mọi thao tác **O(1)**; bộ nhớ **O(k + 1) = O(k)**. Đây là phiên bản kinh điển của hàng đợi vòng (LeetCode 622) nên đáng nhớ; nhược điểm là dễ sai khi quên rằng mảng phải có `k + 1` ô (xem lỗi ở mục 13).

### 7.3. Cách 4 — Danh sách liên kết đôi với hai nút sentinel

**Ý tưởng.** `head` và `tail` là hai nút giả. Phần tử thật nằm giữa chúng. Ta giữ thêm `size` và `cap`.

- `insertFront(v)`: tạo nút mới, nối ngay **sau `head`**.
- `insertLast(v)`: tạo nút mới, nối ngay **trước `tail`**.
- `deleteFront()`: bỏ nút `head.next`. `deleteLast()`: bỏ nút `tail.prev`.
- `getFront()`: `head.next.val`. `getRear()`: `tail.prev.val`.

Để chèn nút `x` vào giữa `a` và `b` (đang kề nhau): `x.prev, x.next = a, b; a.next = x; b.prev = x`. Để xóa `x` nằm giữa `a` và `b`: `a.next = b; b.prev = a`. Nhờ sentinel, `a` và `b` **luôn tồn tại**.

In [11]:
class _Node:
    __slots__ = ("val", "prev", "next")

    def __init__(self, val=0):
        self.val = val
        self.prev = None
        self.next = None


class DequeLinked:
    def __init__(self, k: int):
        self.cap = k
        self.size = 0
        self.head, self.tail = _Node(), _Node()       # hai sentinel
        self.head.next, self.tail.prev = self.tail, self.head

    def _insert_between(self, val, a, b):
        x = _Node(val)
        x.prev, x.next = a, b
        a.next = x
        b.prev = x
        self.size += 1

    def _remove(self, x):
        x.prev.next = x.next
        x.next.prev = x.prev
        self.size -= 1

    def insertFront(self, value: int) -> bool:
        if self.isFull():
            return False
        self._insert_between(value, self.head, self.head.next)
        return True

    def insertLast(self, value: int) -> bool:
        if self.isFull():
            return False
        self._insert_between(value, self.tail.prev, self.tail)
        return True

    def deleteFront(self) -> bool:
        if self.isEmpty():
            return False
        self._remove(self.head.next)
        return True

    def deleteLast(self) -> bool:
        if self.isEmpty():
            return False
        self._remove(self.tail.prev)
        return True

    def getFront(self) -> int:
        return -1 if self.isEmpty() else self.head.next.val

    def getRear(self) -> int:
        return -1 if self.isEmpty() else self.tail.prev.val

    def isEmpty(self) -> bool:
        return self.size == 0

    def isFull(self) -> bool:
        return self.size == self.cap


def view_linked(d):
    out, cur = [], d.head.next
    while cur is not d.tail:
        out.append(cur.val)
        cur = cur.next
    return out


d = DequeLinked(3)
for name, a in zip(ex1[0][1:], ex1[1][1:]):
    r = getattr(d, name)(*a)
    print(f"{name:>12}{tuple(a)!s:>6} -> {str(r):<5} | head <-> {' <-> '.join(map(str, view_linked(d)))} <-> tail")

for names, args, expected in examples:
    assert run_ops(DequeLinked, names, args) == expected
print("OK")

  insertLast  (1,) -> True  | head <-> 1 <-> tail
  insertLast  (2,) -> True  | head <-> 1 <-> 2 <-> tail
 insertFront  (3,) -> True  | head <-> 3 <-> 1 <-> 2 <-> tail
 insertFront  (4,) -> False | head <-> 3 <-> 1 <-> 2 <-> tail
     getRear    () -> 2     | head <-> 3 <-> 1 <-> 2 <-> tail
      isFull    () -> True  | head <-> 3 <-> 1 <-> 2 <-> tail
  deleteLast    () -> True  | head <-> 3 <-> 1 <-> tail
 insertFront  (4,) -> True  | head <-> 4 <-> 3 <-> 1 <-> tail
    getFront    () -> 4     | head <-> 4 <-> 3 <-> 1 <-> tail
OK


**Độ phức tạp cách 4.** Mọi thao tác **O(1)**; bộ nhớ **O(k)** nhưng hằng số lớn hơn (mỗi nút có 2 con trỏ + `val`). Ưu điểm: không cần biết trước kích thước mảng, không có chuyện "chỉ số vòng"; nhược điểm: cấp phát nhiều đối tượng nhỏ.

### 7.4. Cách 5 — Bọc `collections.deque` và tự kiểm tra sức chứa

**Ý tưởng.** `collections.deque` đã có `appendleft`, `append`, `popleft`, `pop` đều `O(1)`. Ta chỉ cần thêm **ràng buộc sức chứa `k`**: trước khi chèn, nếu `len(d) == k` thì trả `False`. (Không dùng `maxlen`, vì khi đầy nó sẽ **âm thầm đẩy phần tử đầu kia ra** thay vì báo `False`.)

Đây là cách ngắn nhất, nhưng trong phỏng vấn thường bị yêu cầu **không** dùng cấu trúc có sẵn.

In [12]:
class DequeBuiltin:
    def __init__(self, k: int):
        self.d = deque()
        self.cap = k

    def insertFront(self, value: int) -> bool:
        if len(self.d) == self.cap:
            return False
        self.d.appendleft(value)
        return True

    def insertLast(self, value: int) -> bool:
        if len(self.d) == self.cap:
            return False
        self.d.append(value)
        return True

    def deleteFront(self) -> bool:
        if not self.d:
            return False
        self.d.popleft()
        return True

    def deleteLast(self) -> bool:
        if not self.d:
            return False
        self.d.pop()
        return True

    def getFront(self) -> int:
        return self.d[0] if self.d else -1

    def getRear(self) -> int:
        return self.d[-1] if self.d else -1

    def isEmpty(self) -> bool:
        return not self.d

    def isFull(self) -> bool:
        return len(self.d) == self.cap


for names, args, expected in examples:
    got = run_ops(DequeBuiltin, names, args)
    print(show(got))
    assert got == expected

[null,true,true,true,false,2,true,true,true,4]
[null,true,-1,-1,false,false,true,7,7,true]


**Độ phức tạp cách 5.** Mọi thao tác **O(1)**; bộ nhớ **O(k)**. (Chú ý: `d[0]` và `d[-1]` là `O(1)`, nhưng `d[i]` ở giữa là `O(n)`.)

**Tóm lại về "code nộp" của từng cách.** Mỗi lớp `DequeList`, `DequeRingSize`, `DequeRingK1`, `DequeLinked`, `DequeBuiltin` ở trên đã đủ 9 phương thức đúng signature của đề; để nộp LeetCode chỉ cần **đổi tên lớp thành `MyCircularDeque`**. Mục 9 làm việc đó cho cách 2.

## 8. TRACE từng bước

### 8.1. Trace bộ đệm vòng (cách 2) trên ví dụ 1

Bảng trạng thái:

| Cột | Ý nghĩa |
|---|---|
| `lệnh` / `kết quả` | lệnh được gọi và giá trị trả về |
| `q` | mảng vòng `k = 3` ô (chữ `_` là ô chưa từng ghi hoặc ô rác) |
| `front`, `size` | con trỏ đầu và số phần tử |
| `đầu→cuối` | nội dung logic của deque |

In [13]:
def trace_ring(names, args):
    """Chạy cách 2 và in trạng thái sau mỗi lệnh (có vẽ vị trí front và ô cuối)."""
    k = args[0][0]
    d = DequeRingSize(k)
    d.q = ["_"] * k                              # để nhìn rõ ô chưa ghi
    print(f"khởi tạo k={k}: q={d.q}, front=0, size=0")
    for step, (name, a) in enumerate(zip(names[1:], args[1:]), 1):
        before = (d.front, d.size)
        r = getattr(d, name)(*a)
        marks = ["   "] * k
        if d.size:
            marks[d.front] = " F "
            rear_i = (d.front + d.size - 1) % k
            marks[rear_i] = " R " if rear_i != d.front else "F=R"
        print(f"{step:>2}. {name}{tuple(a)} -> {r}  (trước: front={before[0]}, size={before[1]})")
        print(f"    q = {d.q}  front={d.front} size={d.size}  đầu→cuối = {logical_view(d.q, d.front, d.size)}")
        print(f"    vị trí:   {marks}")
    return d

_ = trace_ring(*ex1[:2])

khởi tạo k=3: q=['_', '_', '_'], front=0, size=0
 1. insertLast(1,) -> True  (trước: front=0, size=0)
    q = [1, '_', '_']  front=0 size=1  đầu→cuối = [1]
    vị trí:   ['F=R', '   ', '   ']
 2. insertLast(2,) -> True  (trước: front=0, size=1)
    q = [1, 2, '_']  front=0 size=2  đầu→cuối = [1, 2]
    vị trí:   [' F ', ' R ', '   ']
 3. insertFront(3,) -> True  (trước: front=0, size=2)
    q = [1, 2, 3]  front=2 size=3  đầu→cuối = [3, 1, 2]
    vị trí:   ['   ', ' R ', ' F ']
 4. insertFront(4,) -> False  (trước: front=2, size=3)
    q = [1, 2, 3]  front=2 size=3  đầu→cuối = [3, 1, 2]
    vị trí:   ['   ', ' R ', ' F ']
 5. getRear() -> 2  (trước: front=2, size=3)
    q = [1, 2, 3]  front=2 size=3  đầu→cuối = [3, 1, 2]
    vị trí:   ['   ', ' R ', ' F ']
 6. isFull() -> True  (trước: front=2, size=3)
    q = [1, 2, 3]  front=2 size=3  đầu→cuối = [3, 1, 2]
    vị trí:   ['   ', ' R ', ' F ']
 7. deleteLast() -> True  (trước: front=2, size=3)
    q = [1, 2, 3]  front=2 size=2  đầu→cuối 

Quan sát: bước 3 (`insertFront(3)`) lùi `front` từ `0` về `2` — **vòng sang cuối mảng**. Bước 7 (`insertFront(4)` lần hai) lùi `front` từ `2` về `1` và **ghi đè ô 1** (nơi `2` vừa bị `deleteLast` bỏ đi). Mảng chưa bao giờ phải dịch chuyển.

### 8.2. Trace bộ đệm `k + 1` ô (cách 3): nhìn `front` và `rear`

Ta dùng dãy lệnh nhỏ với `k = 2` (mảng 3 ô) để thấy rõ khi nào "rỗng", khi nào "đầy":

In [14]:
def trace_k1(ops, k):
    d = DequeRingK1(k)
    d.q = ["_"] * d.n
    print(f"k={k}, n={d.n}: front={d.front}, rear={d.rear}, rỗng={d.isEmpty()}, đầy={d.isFull()}")
    for name, a in ops:
        r = getattr(d, name)(*a)
        print(f"{name}{tuple(a)} -> {r!s:<5} q={d.q} front={d.front} rear={d.rear} | rỗng={d.isEmpty()} đầy={d.isFull()}")

trace_k1([("insertLast", (1,)), ("insertFront", (2,)), ("insertLast", (3,)),
          ("deleteFront", ()), ("deleteLast", ()), ("deleteLast", ()), ("deleteLast", ())], 2)

k=2, n=3: front=0, rear=0, rỗng=True, đầy=False
insertLast(1,) -> True  q=[1, '_', '_'] front=0 rear=1 | rỗng=False đầy=False
insertFront(2,) -> True  q=[1, '_', 2] front=2 rear=1 | rỗng=False đầy=True
insertLast(3,) -> False q=[1, '_', 2] front=2 rear=1 | rỗng=False đầy=True
deleteFront() -> True  q=[1, '_', 2] front=0 rear=1 | rỗng=False đầy=False
deleteLast() -> True  q=[1, '_', 2] front=0 rear=0 | rỗng=True đầy=False
deleteLast() -> False q=[1, '_', 2] front=0 rear=0 | rỗng=True đầy=False
deleteLast() -> False q=[1, '_', 2] front=0 rear=0 | rỗng=True đầy=False


Đọc trace: với `k = 2`, mảng có 3 ô; sau hai lần chèn thì `(rear + 1) % 3 == front` → **đầy**, và lần chèn thứ ba (`3`) bị từ chối. Khi xóa hết, `front == rear` → **rỗng**; lúc đó `front` và `rear` có thể là bất kỳ ô nào (không nhất thiết là `0`).

### 8.3. Trace danh sách liên kết đôi (cách 4)

Sau mỗi lệnh, in chuỗi nút từ `head` đến `tail`:

In [15]:
def trace_linked(names, args):
    d = DequeLinked(args[0][0])
    print("head <-> tail")
    for step, (name, a) in enumerate(zip(names[1:], args[1:]), 1):
        r = getattr(d, name)(*a)
        chain = " <-> ".join(["head"] + [str(v) for v in view_linked(d)] + ["tail"])
        print(f"{step:>2}. {name}{tuple(a)} -> {r!s:<5} | {chain}")

trace_linked(*ex1[:2])

head <-> tail
 1. insertLast(1,) -> True  | head <-> 1 <-> tail
 2. insertLast(2,) -> True  | head <-> 1 <-> 2 <-> tail
 3. insertFront(3,) -> True  | head <-> 3 <-> 1 <-> 2 <-> tail
 4. insertFront(4,) -> False | head <-> 3 <-> 1 <-> 2 <-> tail
 5. getRear() -> 2     | head <-> 3 <-> 1 <-> 2 <-> tail
 6. isFull() -> True  | head <-> 3 <-> 1 <-> 2 <-> tail
 7. deleteLast() -> True  | head <-> 3 <-> 1 <-> tail
 8. insertFront(4,) -> True  | head <-> 4 <-> 3 <-> 1 <-> tail
 9. getFront() -> 4     | head <-> 4 <-> 3 <-> 1 <-> tail


## 9. Code nộp LeetCode

Chọn **cách 2 (bộ đệm vòng với `front` và `size`)**: mọi thao tác `O(1)`, code ngắn, ít cạm bẫy hơn cách 3 (không có ô "chừa"). Đây là code đúng class/method/signature (trên LeetCode `List` có sẵn; ta không cần import gì thêm):

In [16]:
class MyCircularDeque:

    def __init__(self, k: int):
        self.q = [0] * k          # mảng vòng k ô
        self.front = 0            # chỉ số ô đầu
        self.size = 0             # số phần tử hiện có
        self.cap = k              # sức chứa tối đa

    def insertFront(self, value: int) -> bool:
        if self.isFull():
            return False
        self.front = (self.front - 1) % self.cap      # lùi một ô (vòng tròn)
        self.q[self.front] = value
        self.size += 1
        return True

    def insertLast(self, value: int) -> bool:
        if self.isFull():
            return False
        self.q[(self.front + self.size) % self.cap] = value   # ô trống kế phần tử cuối
        self.size += 1
        return True

    def deleteFront(self) -> bool:
        if self.isEmpty():
            return False
        self.front = (self.front + 1) % self.cap      # tiến một ô
        self.size -= 1
        return True

    def deleteLast(self) -> bool:
        if self.isEmpty():
            return False
        self.size -= 1
        return True

    def getFront(self) -> int:
        return -1 if self.isEmpty() else self.q[self.front]

    def getRear(self) -> int:
        return -1 if self.isEmpty() else self.q[(self.front + self.size - 1) % self.cap]

    def isEmpty(self) -> bool:
        return self.size == 0

    def isFull(self) -> bool:
        return self.size == self.cap


# Your MyCircularDeque object will be instantiated and called as such:
# obj = MyCircularDeque(k)
# param_1 = obj.insertFront(value)
# ...

for names, args, expected in examples:
    got = run_ops(MyCircularDeque, names, args)
    print(show(got))
    assert got == expected

[null,true,true,true,false,2,true,true,true,4]
[null,true,-1,-1,false,false,true,7,7,true]


## 10. Vì sao thuật toán đúng

**Bất biến của cách 2.** Tại mọi thời điểm: các phần tử của deque (đầu→cuối) nằm ở các ô `front, front+1, ..., front+size-1` (mọi chỉ số lấy `% cap`), và `0 ≤ size ≤ cap`. Ta chứng minh mỗi lệnh bảo toàn bất biến:

- **`insertFront`** (khi `size < cap`): ô `(front - 1) % cap` **không thuộc** đoạn đang dùng, vì đoạn dùng có `size < cap` ô liên tiếp bắt đầu từ `front` nên ô ngay trước `front` còn trống (trừ khi `size == cap`, đã bị loại). Ghi `value` vào đó và đặt `front` mới = ô đó, `size + 1` → đoạn mới là `value` rồi các phần tử cũ — đúng thứ tự đầu→cuối.
- **`insertLast`**: ô `(front + size) % cap` là ô ngay sau phần tử cuối; nó trống vì `size < cap`. Ghi vào đó và `size + 1` → `value` thành phần tử cuối.
- **`deleteFront`**: bỏ phần tử ở `front`: tiến `front` một ô, `size - 1` → đoạn còn lại đúng là các phần tử sau.
- **`deleteLast`**: bỏ phần tử ở ô `(front + size - 1) % cap` bằng cách `size - 1`.
- **`getFront`/`getRear`** đọc đúng ô đầu và ô cuối nhờ bất biến; khi `size == 0` trả `-1` theo đề.
- **`isEmpty`/`isFull`** là `size == 0` và `size == cap`, chính xác theo định nghĩa (không còn mơ hồ vì có biến `size`).

**Vì sao `% cap` không làm hai phần tử đè nhau.** Đoạn dùng có `size ≤ cap` ô liên tiếp trên vòng nên các ô đôi một khác nhau.

**Cách 3** đúng vì mảng `k + 1` ô: số phần tử = `(rear - front) mod (k+1)` luôn trong `[0, k]`, nên `front == rear` chỉ xảy ra khi có `0` phần tử (đầy là `k` phần tử ⇒ `(rear+1) % n == front`). **Cách 4** đúng nhờ bất biến "dây nút nối đúng thứ tự đầu→cuối, giữa hai sentinel". **Cách 1 và 5** đúng hiển nhiên (là list/deque thật), nên ta dùng chúng làm mô hình tham chiếu.

## 11. Phân tích Time / Space Complexity

Gọi `k` là sức chứa, `m` là số lệnh.

| Cách | `insertFront` / `deleteFront` | Các lệnh khác | Space | Ghi chú |
|---|---|---|---|---|
| 1. `list` + `insert(0)`/`pop(0)` | O(k) | O(1) | O(k) | dễ nhất, dùng làm oracle |
| 2. Vòng `front` + `size` | O(1) | O(1) | O(k) | bản nộp |
| 3. Vòng `k + 1` ô, `front`/`rear` | O(1) | O(1) | O(k) | kinh điển; nhớ chừa 1 ô |
| 4. Liên kết đôi + sentinel | O(1) | O(1) | O(k) | hằng số lớn hơn |
| 5. `collections.deque` + kiểm tra `len` | O(1) | O(1) | O(k) | ngắn nhất, thường bị cấm khi phỏng vấn |

Tổng thời gian cho `m` lệnh: `O(m)` với cách 2–5; `O(m·k)` ở trường hợp xấu của cách 1.

**Vì sao `deleteLast` chỉ cần `size -= 1`?** Ô cuối cũ không bị xóa dữ liệu nhưng nằm ngoài đoạn `[front, front + size)` nên coi như trống; lần `insertLast` sau sẽ ghi đè. Tương tự `deleteFront` chỉ dời `front`.

Ràng buộc `k ≤ 1000`, `2000` lệnh nên mọi cách đều rất nhanh; điều ta học là **thiết kế đúng độ phức tạp**.

## 12. Bộ test

Ta kiểm tra theo nhiều lớp:

1. Ví dụ của đề cho cả 5 cách.
2. Edge case: `k = 1`, thao tác trên deque rỗng, đầy rồi xóa hết rồi đầy lại, nhiều vòng quấn quanh.
3. **Phản ví dụ** của các cài đặt sai thường gặp (xem thêm mục 13).
4. Đối chiếu 5 cách với **mô hình tham chiếu độc lập** (một list Python với phép cắt lát, viết khác cách 1) bằng các chuỗi lệnh ngẫu nhiên có seed, đúng ràng buộc.
5. Test lớn (2000 lệnh, `k = 1000`) và so sánh thời gian.

In [17]:
class RefDeque:
    """Mô hình tham chiếu độc lập, cố tình viết khác cách 1: dựng lại list bằng phép nối lát mỗi lần."""
    def __init__(self, k):
        self.items = ()
        self.k = k

    def insertFront(self, v):
        if len(self.items) >= self.k:
            return False
        self.items = (v,) + self.items
        return True

    def insertLast(self, v):
        if len(self.items) >= self.k:
            return False
        self.items = self.items + (v,)
        return True

    def deleteFront(self):
        if not self.items:
            return False
        self.items = self.items[1:]
        return True

    def deleteLast(self):
        if not self.items:
            return False
        self.items = self.items[:-1]
        return True

    def getFront(self):
        return self.items[0] if self.items else -1

    def getRear(self):
        return self.items[-1] if self.items else -1

    def isEmpty(self):
        return len(self.items) == 0

    def isFull(self):
        return len(self.items) >= self.k


impls = {
    "list": DequeList,
    "ring_size": DequeRingSize,
    "ring_k1": DequeRingK1,
    "linked": DequeLinked,
    "builtin": DequeBuiltin,
    "final": MyCircularDeque,
}

for names, args, expected in examples:
    assert run_ops(RefDeque, names, args) == expected
    for name, cls in impls.items():
        assert run_ops(cls, names, args) == expected, name
print("tất cả cách khớp ví dụ đề:", list(impls))

tất cả cách khớp ví dụ đề: ['list', 'ring_size', 'ring_k1', 'linked', 'builtin', 'final']


In [18]:
# --- Edge case: dựng chuỗi lệnh bằng hàm nhỏ cho dễ đọc ---
def script(k, ops):
    names = ["MyCircularDeque"] + [o[0] for o in ops]
    args = [[k]] + [list(o[1:]) for o in ops]
    return names, args


cases = {
    "k=1 đủ thứ": script(1, [("isEmpty",), ("insertFront", 5), ("isFull",), ("insertLast", 6), ("insertFront", 7),
                             ("getFront",), ("getRear",), ("deleteLast",), ("deleteFront",), ("getFront",), ("insertLast", 9),
                             ("getRear",), ("deleteFront",), ("isEmpty",)]),
    "xóa khi rỗng": script(3, [("deleteFront",), ("deleteLast",), ("getFront",), ("getRear",), ("isEmpty",), ("isFull",)]),
    "đầy rồi trống rồi đầy": script(3, [("insertLast", 1), ("insertLast", 2), ("insertLast", 3), ("insertLast", 4), ("isFull",),
                                       ("deleteFront",), ("deleteFront",), ("deleteFront",), ("isEmpty",), ("deleteFront",),
                                       ("insertFront", 7), ("insertFront", 8), ("insertFront", 9), ("insertFront", 10),
                                       ("getFront",), ("getRear",)]),
    "quấn vòng nhiều lần": script(2, [("insertFront", 1), ("deleteLast",)] * 6 + [("insertLast", 4), ("insertFront", 3), ("getFront",), ("getRear",)]),
    "chèn đầu liên tục": script(4, [("insertFront", i) for i in range(6)] + [("getFront",), ("getRear",), ("deleteLast",), ("getRear",)]),
    "chèn cuối liên tục": script(4, [("insertLast", i) for i in range(6)] + [("getFront",), ("getRear",), ("deleteFront",), ("getFront",)]),
    "giá trị biên 0 và 1000": script(3, [("insertLast", 0), ("insertFront", 1000), ("getFront",), ("getRear",)]),
}
for title, (names, args) in cases.items():
    want = run_ops(RefDeque, names, args)
    for name, cls in impls.items():
        got = run_ops(cls, names, args)
        assert got == want, (title, name, got, want)
    print(f"{title:<24}", show(want))

k=1 đủ thứ               [null,true,true,true,false,false,5,5,true,false,-1,true,9,true,true]
xóa khi rỗng             [null,false,false,-1,-1,true,false]
đầy rồi trống rồi đầy    [null,true,true,true,false,true,true,true,true,true,false,true,true,true,false,9,7]
quấn vòng nhiều lần      [null,true,true,true,true,true,true,true,true,true,true,true,true,true,true,3,4]
chèn đầu liên tục        [null,true,true,true,true,false,false,3,0,true,1]
chèn cuối liên tục       [null,true,true,true,true,false,false,0,3,true,1]
giá trị biên 0 và 1000   [null,true,true,1000,0]


**Phản ví dụ.** Các cài đặt "trông có vẻ đúng" nhưng sai. Ta chứng minh bằng chính ví dụ 1 của đề:

- *Sai A:* bộ đệm vòng `k` ô chỉ với `front`/`rear` (bỏ `size`, không chừa ô) — sức chứa thực tế chỉ còn `k - 1`.
- *Sai B:* `getRear` dùng `(front + size) % cap` (lệch một ô).
- *Sai C:* `deleteFront` quên tiến `front`.

In [19]:
class WrongA:                      # mảng k ô, front/rear, "đầy" khi (rear+1)%k == front  -> chỉ chứa k-1
    def __init__(self, k):
        self.q, self.cap, self.front, self.rear = [0] * k, k, 0, 0

    def isEmpty(self):
        return self.front == self.rear

    def isFull(self):
        return (self.rear + 1) % self.cap == self.front

    def insertFront(self, v):
        if self.isFull():
            return False
        self.front = (self.front - 1) % self.cap
        self.q[self.front] = v
        return True

    def insertLast(self, v):
        if self.isFull():
            return False
        self.q[self.rear] = v
        self.rear = (self.rear + 1) % self.cap
        return True

    def deleteFront(self):
        if self.isEmpty():
            return False
        self.front = (self.front + 1) % self.cap
        return True

    def deleteLast(self):
        if self.isEmpty():
            return False
        self.rear = (self.rear - 1) % self.cap
        return True

    def getFront(self):
        return -1 if self.isEmpty() else self.q[self.front]

    def getRear(self):
        return -1 if self.isEmpty() else self.q[(self.rear - 1) % self.cap]


class WrongB(DequeRingSize):       # getRear lệch một ô
    def getRear(self):
        return -1 if self.isEmpty() else self.q[(self.front + self.size) % self.cap]


class WrongC(DequeRingSize):       # deleteFront quên tiến front
    def deleteFront(self):
        if self.isEmpty():
            return False
        self.size -= 1
        return True


names, args, expected = ex1
print("đúng :", show(expected))
for label, cls in (("sai A", WrongA), ("sai B", WrongB), ("sai C", WrongC)):
    got = run_ops(cls, names, args)
    print(label, ":", show(got), "| khớp?", got == expected)
    if label != "sai C":
        assert got != expected, label

# Sai C lộ ra ở một chuỗi lệnh khác (ví dụ 1 không có deleteFront):
n2, a2 = script(3, [("insertLast", 1), ("insertLast", 2), ("deleteFront",), ("getFront",)])
print("sai C trên [1,2], deleteFront, getFront:", run_ops(WrongC, n2, a2), "đúng:", run_ops(MyCircularDeque, n2, a2))
assert run_ops(WrongC, n2, a2) != run_ops(MyCircularDeque, n2, a2)

đúng : [null,true,true,true,false,2,true,true,true,4]
sai A : [null,true,true,false,false,2,true,true,true,4] | khớp? False
sai B : [null,true,true,true,false,3,true,true,true,4] | khớp? False
sai C : [null,true,true,true,false,2,true,true,true,4] | khớp? True
sai C trên [1,2], deleteFront, getFront: [None, True, True, True, 1] đúng: [None, True, True, True, 2]


Giải thích: **A** cho `insertFront(3)` thất bại (`false`) vì với `k = 3` chỉ chứa được `2` phần tử; **B** cho `getRear` trả giá trị ở ô *sau* ô cuối (rác `0` hoặc phần tử đầu); **C** sau `deleteFront` vẫn nhìn ô cũ nên `getFront` trả `1` thay vì `2`.

**Đối chiếu nhiều chuỗi lệnh ngẫu nhiên (có seed, đúng ràng buộc).** Ta sinh chuỗi lệnh gồm 8 loại thao tác với tỉ lệ nghiêng về chèn/xóa để deque thường xuyên đầy/rỗng; `k` nhỏ để hay quấn vòng, và có cả `k` lớn.

In [20]:
OPS_INS = ["insertFront", "insertLast"]
OPS_DEL = ["deleteFront", "deleteLast"]
OPS_GET = ["getFront", "getRear", "isEmpty", "isFull"]


def random_script(rng, k, n_ops, bias):
    """bias in {'insert','delete','mixed'}: nghiêng về loại thao tác nào."""
    weights = {"insert": (6, 2, 2), "delete": (2, 6, 2), "mixed": (4, 4, 2)}[bias]
    ops = []
    for _ in range(n_ops):
        kind = rng.choices(["ins", "del", "get"], weights)[0]
        if kind == "ins":
            ops.append((rng.choice(OPS_INS), rng.randint(0, 1000)))
        elif kind == "del":
            ops.append((rng.choice(OPS_DEL),))
        else:
            ops.append((rng.choice(OPS_GET),))
    return script(k, ops)


rng = random.Random(641)
count = 0
for _ in range(500):
    k = rng.choice([1, 1, 2, 3, 4, 5, 8, 20])
    n_ops = rng.randint(1, 80)
    names, args = random_script(rng, k, n_ops, rng.choice(["insert", "delete", "mixed"]))
    want = run_ops(RefDeque, names, args)
    for name, cls in impls.items():
        got = run_ops(cls, names, args)
        assert got == want, (name, k, names[:12])
    count += 1
print("đối chiếu", count, "chuỗi lệnh ngẫu nhiên (k nhỏ):", len(impls), "cài đặt khớp mô hình tham chiếu")

# k lớn, đúng ràng buộc: k <= 1000, tối đa 2000 lệnh
for _ in range(20):
    k = rng.randint(100, 1000)
    names, args = random_script(rng, k, 1999, rng.choice(["insert", "mixed"]))
    assert len(names) <= 2000
    want = run_ops(RefDeque, names, args)
    for name, cls in impls.items():
        assert run_ops(cls, names, args) == want, (name, k)
print("20 chuỗi 2000 lệnh với k lớn: OK")

đối chiếu 500 chuỗi lệnh ngẫu nhiên (k nhỏ): 6 cài đặt khớp mô hình tham chiếu


20 chuỗi 2000 lệnh với k lớn: OK


In [21]:
# Bất biến nội bộ của cách 2 sau mỗi lệnh (kiểm tra thêm cho bản nộp)
rng = random.Random(6410)
for _ in range(200):
    k = rng.randint(1, 10)
    d, ref = MyCircularDeque(k), RefDeque(k)
    names, args = random_script(rng, k, 60, "mixed")
    for nm, a in zip(names[1:], args[1:]):
        getattr(d, nm)(*a)
        getattr(ref, nm)(*a)
        assert 0 <= d.size <= d.cap and 0 <= d.front < d.cap
        assert logical_view(d.q, d.front, d.size) == list(ref.items)
print("bất biến (front trong [0, cap), 0 <= size <= cap, nội dung đúng thứ tự) luôn đúng")

# So sánh thời gian: list (O(k) ở đầu) và vòng (O(1)), cùng chuỗi lệnh chèn/xóa đầu
names, args = script(1000, [("insertFront", 1)] * 1000 + [("deleteFront",)] * 1000)
for label, cls in (("list", DequeList), ("ring_size", DequeRingSize), ("linked", DequeLinked), ("builtin", DequeBuiltin)):
    t0 = time.perf_counter()
    for _ in range(30):
        run_ops(cls, names, args)
    print(f"{label:>10}: {(time.perf_counter() - t0) * 1000:8.2f} ms cho 30 lần x 2000 lệnh")

bất biến (front trong [0, cap), 0 <= size <= cap, nội dung đúng thứ tự) luôn đúng
      list:    29.01 ms cho 30 lần x 2000 lệnh
 ring_size:    27.60 ms cho 30 lần x 2000 lệnh
    linked:    18.73 ms cho 30 lần x 2000 lệnh
   builtin:     9.34 ms cho 30 lần x 2000 lệnh


## 13. Các lỗi người mới hay gặp

### Lỗi 1: `front == rear` vừa là rỗng vừa là đầy

Nếu dùng mảng `k` ô với `front`/`rear` và `isEmpty` là `front == rear`, thì để không nhầm với đầy phải coi `(rear + 1) % k == front` là đầy — và sức chứa tụt xuống `k - 1`. Đó chính là `WrongA` ở mục 12.

In [22]:
names, args = script(3, [("insertLast", 1), ("insertLast", 2), ("insertLast", 3), ("isFull",)])
print("sai  (k=3, 3 lần insertLast):", run_ops(WrongA, names, args))
print("đúng                        :", run_ops(MyCircularDeque, names, args))

sai  (k=3, 3 lần insertLast): [None, True, True, False, True]
đúng                        : [None, True, True, True, True]


**Vì sao sai.** Với `k = 3` ta phải chứa được `3` phần tử, nhưng bản sai báo đầy khi mới có `2` nên lần chèn thứ ba trả `False`. **Cách sửa:** hoặc thêm biến `size`, hoặc cấp mảng `k + 1` ô (cách 3).

### Lỗi 2: `getRear` lệch một ô

Ô cuối là `(front + size - 1) % cap`, còn `(front + size) % cap` là **ô trống kế sau**. Dùng nhầm sẽ trả phần tử rác hoặc phần tử đầu.

In [23]:
names, args = script(3, [("insertLast", 1), ("insertLast", 2), ("getRear",)])
print("sai  :", run_ops(WrongB, names, args))        # đọc ô trống: 0
print("đúng :", run_ops(MyCircularDeque, names, args))

sai  : [None, True, True, 0]
đúng : [None, True, True, 2]


**Vì sao sai.** Sau hai lần chèn cuối: `q = [1, 2, 0]`, `front = 0`, `size = 2`. Ô cuối là `(0 + 2 - 1) % 3 = 1` (giá trị `2`). Bản sai đọc ô `(0 + 2) % 3 = 2` (giá trị rác `0`). **Cách sửa:** nhớ "ô cuối = ô trống kế sau **trừ một**".

### Lỗi 3: `deleteFront` quên tiến `front`

Chỉ giảm `size` mà không dời `front` thì phần tử bị xóa vẫn là "đầu", còn phần tử cuối thật bị mất. (Đó là `WrongC`.)

In [24]:
names, args = script(3, [("insertLast", 1), ("insertLast", 2), ("deleteFront",), ("getFront",), ("getRear",)])
print("sai  :", run_ops(WrongC, names, args))       # getFront vẫn 1, getRear = 1 (sai)
print("đúng :", run_ops(MyCircularDeque, names, args))

sai  : [None, True, True, True, 1, 1]
đúng : [None, True, True, True, 2, 2]


**Vì sao sai.** Sau `deleteFront` đúng ra deque còn `[2]`; bản sai giữ `front = 0` và `size = 1` nên nhìn thấy `[1]`. **Cách sửa:** `deleteFront` luôn là `front = (front + 1) % cap` **và** `size -= 1`.

### Lỗi 4: trả giá trị rác thay vì `-1` khi rỗng

Quên kiểm tra rỗng trong `getFront`/`getRear`: mảng vẫn giữ giá trị cũ (hoặc `0` khởi tạo) nên trả về một số hợp lệ nhưng sai.

In [25]:
class WrongD(DequeRingSize):
    def getFront(self):
        return self.q[self.front]                  # SAI: không kiểm tra rỗng

names, args = script(2, [("insertLast", 9), ("deleteLast",), ("getFront",)])
print("sai  :", run_ops(WrongD, names, args), "(deque rỗng nhưng trả 9)")
print("đúng :", run_ops(MyCircularDeque, names, args))

sai  : [None, True, True, 9] (deque rỗng nhưng trả 9)
đúng : [None, True, True, -1]


**Vì sao sai.** `deleteLast` chỉ giảm `size`, **không xóa dữ liệu** khỏi mảng; ô cũ vẫn chứa `9`. **Cách sửa:** luôn kiểm tra `isEmpty()` trước khi đọc.

### Lỗi 5: quên `% cap` khi lùi/tiến, chỉ số trôi dần ra ngoài mảng

Viết `self.front -= 1` (không `% cap`). Trong Python chỉ số âm nhỏ vẫn "trúng" nhờ truy cập từ cuối mảng, nên lỗi **che giấu** lúc đầu, rồi bùng nổ khi `front` trôi quá `-cap`.

In [26]:
class WrongE(DequeRingSize):
    def insertFront(self, value):
        if self.isFull():
            return False
        self.front -= 1                            # SAI: thiếu % cap
        self.q[self.front] = value
        self.size += 1
        return True

names, args = script(2, [("insertFront", 1), ("deleteLast",)] * 3)
try:
    print(run_ops(WrongE, names, args))
except IndexError as e:
    print("IndexError:", e)
print("đúng :", run_ops(MyCircularDeque, names, args))

IndexError: list assignment index out of range
đúng : [None, True, True, True, True, True, True]


**Vì sao sai.** Mỗi vòng `insertFront; deleteLast` làm `front` giảm một: `-1, -2, -3`; với `cap = 2` thì `q[-3]` ra ngoài mảng. **Cách sửa:** luôn dùng `(front - 1) % cap`.

### Lỗi 6: dùng `deque(maxlen=k)` và tưởng nó "từ chối" khi đầy

`deque(maxlen=k).append(x)` khi đầy **âm thầm bỏ phần tử ở đầu kia**, không trả `False`. Bài này đòi `False`, nên phải tự kiểm tra `len`.

In [27]:
m = deque(maxlen=3)
for v in (1, 2, 3):
    m.append(v)
m.append(4)                                    # không báo lỗi! 1 bị đẩy ra
print(m)                                       # deque([2, 3, 4], maxlen=3)

deque([2, 3, 4], maxlen=3)


**Cách sửa.** Dùng `deque()` không `maxlen` và kiểm tra `len(d) == k` trước khi chèn (như `DequeBuiltin`).

## 14. Mẹo & nhận dạng pattern

- **"Thêm/xóa ở hai đầu, sức chứa cố định"** → **ring buffer** (mảng + chỉ số vòng `% k`). Đây là mô hình nền của hàng đợi vòng (622), deque vòng (641), bộ đệm log, bộ đệm âm thanh.
- **Trạng thái tối thiểu**: `front` + `size` (+ `cap`). Mọi vị trí khác suy ra bằng công thức; nhớ bảng: ô đầu `front`, ô cuối `(front + size - 1) % cap`, ô trống kế cuối `(front + size) % cap`, ô trống kế đầu `(front - 1) % cap`.
- **Phân biệt rỗng/đầy**: biến `size`, hoặc chừa một ô, hoặc một cờ boolean. Chọn một và bám theo.
- **Thiết kế lớp, tự kiểm**: dựng bộ chạy chuỗi lệnh (`run_ops`) và **mô hình tham chiếu đơn giản** (list/`deque`), rồi so với cài đặt của mình trên chuỗi lệnh ngẫu nhiên có seed.
- **Danh sách liên kết đôi + sentinel** triệt tiêu các trường hợp biên "rỗng/một nút".
- **Nghĩ đến `collections.deque`** khi làm bài thật (BFS, cửa sổ trượt), nhưng biết tự cài để trả lời phỏng vấn.
- Chú ý **đề đòi trả `-1` hay `False`** ở từng lệnh; copy đúng từng chữ.

## 15. Biến thể

### 15.1. Hàng đợi vòng một đầu (LeetCode 622 Design Circular Queue)

Chỉ cho `enQueue` ở cuối và `deQueue` ở đầu; `Front`, `Rear`, `isEmpty`, `isFull`. Dùng chính ring buffer ở trên.

In [28]:
class MyCircularQueue:
    def __init__(self, k: int):
        self.q, self.cap, self.front, self.size = [0] * k, k, 0, 0

    def enQueue(self, value: int) -> bool:
        if self.size == self.cap:
            return False
        self.q[(self.front + self.size) % self.cap] = value
        self.size += 1
        return True

    def deQueue(self) -> bool:
        if self.size == 0:
            return False
        self.front = (self.front + 1) % self.cap
        self.size -= 1
        return True

    def Front(self) -> int:
        return -1 if self.size == 0 else self.q[self.front]

    def Rear(self) -> int:
        return -1 if self.size == 0 else self.q[(self.front + self.size - 1) % self.cap]

    def isEmpty(self) -> bool:
        return self.size == 0

    def isFull(self) -> bool:
        return self.size == self.cap


names = ["MyCircularQueue", "enQueue", "enQueue", "enQueue", "enQueue", "Rear", "isFull", "deQueue", "enQueue", "Rear"]
args = [[3], [1], [2], [3], [4], [], [], [], [4], []]
res = run_ops(MyCircularQueue, names, args)
print(show(res))
assert res == [None, True, True, True, False, 3, True, True, True, 4]

[null,true,true,true,false,3,true,true,true,4]


### 15.2. Duyệt nội dung theo thứ tự (`to_list`) và `__len__`

Thêm tiện ích để debug: trả về nội dung đầu→cuối.

In [29]:
class RingDequeWithToList(MyCircularDeque):
    def to_list(self):
        return [self.q[(self.front + i) % self.cap] for i in range(self.size)]

    def __len__(self):
        return self.size


d = RingDequeWithToList(4)
for v in (1, 2):
    d.insertLast(v)
for v in (3, 4):
    d.insertFront(v)
print(d.to_list(), len(d))                      # [4, 3, 1, 2] 4
assert d.to_list() == [4, 3, 1, 2] and len(d) == 4

[4, 3, 1, 2] 4


### 15.3. Deque **tự giãn** (không giới hạn sức chứa)

Khi đầy, cấp mảng gấp đôi, chép phần tử theo thứ tự đầu→cuối vào đầu mảng mới, đặt `front = 0`. Mỗi lần chèn là `O(1)` **khấu hao** (amortized).

In [30]:
class GrowingDeque:
    def __init__(self, cap=2):
        self.q, self.cap, self.front, self.size = [0] * cap, cap, 0, 0

    def _grow(self):
        new = [self.q[(self.front + i) % self.cap] for i in range(self.size)]
        self.q = new + [0] * (self.cap)                # gấp đôi
        self.cap *= 2
        self.front = 0

    def append(self, v):
        if self.size == self.cap:
            self._grow()
        self.q[(self.front + self.size) % self.cap] = v
        self.size += 1

    def appendleft(self, v):
        if self.size == self.cap:
            self._grow()
        self.front = (self.front - 1) % self.cap
        self.q[self.front] = v
        self.size += 1

    def to_list(self):
        return [self.q[(self.front + i) % self.cap] for i in range(self.size)]


g, ref = GrowingDeque(), deque()
rng = random.Random(15)
for _ in range(500):
    v = rng.randint(0, 99)
    if rng.random() < 0.5:
        g.append(v); ref.append(v)
    else:
        g.appendleft(v); ref.appendleft(v)
assert g.to_list() == list(ref)
print("giãn đến cap =", g.cap, "cho", g.size, "phần tử: OK")

giãn đến cap = 512 cho 500 phần tử: OK


### 15.4. Cửa sổ trượt lớn nhất (LeetCode 239): deque đơn điệu

Một ứng dụng kinh điển của deque: giữ các **chỉ số** theo giá trị giảm dần; phần tử lớn nhất của cửa sổ luôn ở **đầu**.

In [31]:
def max_sliding_window(nums, k):
    dq, out = deque(), []                       # dq chứa chỉ số, nums[dq] giảm dần
    for i, x in enumerate(nums):
        while dq and nums[dq[-1]] <= x:
            dq.pop()                            # bỏ các phần tử nhỏ hơn x ở cuối
        dq.append(i)
        if dq[0] <= i - k:
            dq.popleft()                        # đầu đã ra khỏi cửa sổ
        if i >= k - 1:
            out.append(nums[dq[0]])
    return out

nums = [1, 3, -1, -3, 5, 3, 6, 7]
print(max_sliding_window(nums, 3))              # [3, 3, 5, 5, 6, 7]
assert max_sliding_window(nums, 3) == [3, 3, 5, 5, 6, 7]
assert max_sliding_window(nums, 3) == [max(nums[i:i + 3]) for i in range(len(nums) - 2)]

[3, 3, 5, 5, 6, 7]


### 15.5. Biến thể để tự nghĩ

- **Bộ đệm ghi đè** (ring buffer kiểu log): khi đầy thì `insertLast` **ghi đè phần tử đầu** thay vì báo `False`.
- **Hàng đợi ưu tiên hai đầu** (min và max cùng lúc).
- **Deque an toàn đa luồng**: cần khóa/CAS cho `front`, `size`.

## 16. So sánh các cách làm

| Cách | Ý tưởng | Time mỗi lệnh | Space | Khi nào dùng |
|---|---|---|---|---|
| 1. `list` + `insert(0)`/`pop(0)` | dịch phần tử ở đầu | O(k) ở đầu, O(1) cuối | O(k) | làm oracle/hiểu đề; `k` nhỏ |
| 2. Vòng `front` + `size` | đổi con trỏ, `% cap`, `size` phân biệt rỗng/đầy | O(1) | O(k) | **cách nộp mặc định**, ít lỗi nhất |
| 3. Vòng `k + 1` ô | `front`/`rear`, chừa 1 ô | O(1) | O(k) | khi đề cấm biến `size`; kinh điển của queue vòng |
| 4. Liên kết đôi + sentinel | nút `prev`/`next`, hai đầu cố định | O(1) | O(k), hằng số lớn | khi cần chèn/xóa ở **giữa** hoặc kích thước linh hoạt |
| 5. `collections.deque` | thư viện có sẵn + kiểm tra `len` | O(1) | O(k) | thực chiến, không phải phỏng vấn "tự cài" |

**Lời khuyên.** Nắm cách 2 và cách 4 (hai cấu trúc kinh điển: mảng vòng và liên kết đôi), biết cách 3 vì hay gặp trong tài liệu, dùng cách 5 khi viết code thật.

## 17. 5 bài tập vận dụng (dễ → khó)

Cell **TODO** chỉ gán `None` hoặc định nghĩa hàm rỗng, **không được gọi** khi chạy notebook. Đáp án ở mục 19.

**Bài 1 (đoán output).** Dự đoán `k = 2`: kết quả của từng lệnh rồi chạy:

In [32]:
names = ["MyCircularDeque", "insertFront", "insertLast", "insertLast", "getFront", "getRear", "deleteFront", "getFront", "isFull"]
args = [[2], [5], [6], [7], [], [], [], [], []]
print(show(run_ops(MyCircularDeque, names, args)))

[null,true,true,false,5,6,true,6,false]


**Bài 2 (tính tay).** Với bộ đệm vòng **cách 2**, `k = 4` (`q = [0,0,0,0]`, `front = 0`, `size = 0`), thực hiện lần lượt: `insertLast(1)`, `insertLast(2)`, `insertFront(3)`, `deleteLast()`, `insertFront(4)`. Tính trạng thái cuối và gán `ans2 = (front, size, q)` (đang là `None`).

In [33]:
ans2 = None      # TODO: tuple (front, size, danh sách q 4 phần tử)

**Bài 3 (điền TODO).** Viết `rear_index(front, size, cap)` trả về **chỉ số ô cuối** của deque trong mảng vòng (giả sử `size >= 1`). Ví dụ `rear_index(3, 4, 5) == 1`.

In [34]:
def rear_index(front, size, cap):
    # TODO: ô cuối = ô đầu đi tiến (size - 1) bước trên vòng tròn
    pass

**Bài 4 (tự viết lại).** Không nhìn mục 7, tự viết lớp `MyDeque` (cùng 9 phương thức như đề) bằng **danh sách liên kết đôi có sentinel**, hoặc bằng bộ đệm vòng `k + 1` ô nếu bạn thích.

In [35]:
class MyDeque:
    # TODO: __init__(self, k) và 8 phương thức còn lại
    pass

**Bài 5 (mini challenge).** (a) Viết `MyCircularQueue2` cho bài 622 bằng bộ đệm vòng **`k + 1` ô** (không dùng `size`). (b) Viết lớp `OverwriteBuffer(k)` có `push(x)` (thêm vào cuối; nếu đầy thì **ghi đè phần tử cũ nhất**) và `to_list()` (đầu→cuối).

In [36]:
class MyCircularQueue2:
    # TODO
    pass

class OverwriteBuffer:
    # TODO
    pass

## 18. Gợi ý 3 mức cho bài tập

**Mức 1 (nhẹ):**
- Bài 1: nhớ deque tối đa 2 phần tử; lệnh thứ ba `insertLast(7)` sẽ đầy.
- Bài 2: `insertLast` ghi ô `(front + size) % cap`; `insertFront` lùi `front` rồi ghi.
- Bài 3: ô đầu là `front`.
- Bài 4: mỗi nút có `val`, `prev`, `next`.
- Bài 5: (b) khi đầy, "ghi đè phần tử cũ nhất" nghĩa là gì với `front`?

**Mức 2 (vừa):**
- Bài 2: sau `insertLast(1), insertLast(2)`: `q = [1,2,0,0]`; `insertFront(3)` ghi ô `3`.
- Bài 3: `(front + size - 1) % cap`.
- Bài 4: dùng `head`/`tail` giả, `head.next`/`tail.prev` là hai đầu thật.
- Bài 5: (a) rỗng `front == rear`, đầy `(rear + 1) % n == front`, `n = k + 1`; (b) khi đầy: ghi vào ô `(front + size) % cap` (chính là ô `front`) rồi tiến `front`, không tăng `size`.

**Mức 3 (gần đáp án):**
- Bài 3: xem mục 3.3.
- Bài 4: xem cách 4 (mục 7.3) hoặc cách 3 (mục 7.2).
- Bài 5: (a) xem cách 3 nhưng chỉ giữ `enQueue/deQueue/Front/Rear`; (b) `if size < cap: ghi ở (front+size)%cap; size += 1; else: q[front] = x; front = (front+1)%cap`.

## 19. Đáp án tham khảo

Chạy kiểm tra bằng `assert`.

In [37]:
# Bài 1: k=2: insertFront(5) T; insertLast(6) T (đầy: [5,6]); insertLast(7) F; getFront 5; getRear 6; deleteFront T; getFront 6; isFull F
names = ["MyCircularDeque", "insertFront", "insertLast", "insertLast", "getFront", "getRear", "deleteFront", "getFront", "isFull"]
args = [[2], [5], [6], [7], [], [], [], [], []]
assert run_ops(MyCircularDeque, names, args) == [None, True, True, False, 5, 6, True, 6, False]

# Bài 2: tính tay ra q = [1, 2, 4, 3], front = 2, size = 3
ans2 = (2, 3, [1, 2, 4, 3])
d = MyCircularDeque(4)
d.insertLast(1); d.insertLast(2); d.insertFront(3); d.deleteLast(); d.insertFront(4)
assert ans2 == (d.front, d.size, d.q)
print("Bài 1, 2 OK")

Bài 1, 2 OK


In [38]:
# Bài 3
def rear_index(front, size, cap):
    return (front + size - 1) % cap

assert rear_index(3, 4, 5) == 1
assert rear_index(0, 1, 5) == 0
assert rear_index(4, 2, 5) == 0
assert rear_index(2, 3, 3) == 1
for cap in range(1, 8):
    for front in range(cap):
        for size in range(1, cap + 1):
            d = DequeRingSize(cap)
            d.front, d.size = front, size
            assert d.getRear() == d.q[rear_index(front, size, cap)]
print("Bài 3 OK")

Bài 3 OK


In [39]:
# Bài 4 (bản bộ đệm vòng k+1 ô; có thể thay bằng liên kết đôi)
class MyDeque:
    def __init__(self, k):
        self.n = k + 1
        self.q = [0] * self.n
        self.front = self.rear = 0

    def isEmpty(self):
        return self.front == self.rear

    def isFull(self):
        return (self.rear + 1) % self.n == self.front

    def insertFront(self, v):
        if self.isFull():
            return False
        self.front = (self.front - 1) % self.n
        self.q[self.front] = v
        return True

    def insertLast(self, v):
        if self.isFull():
            return False
        self.q[self.rear] = v
        self.rear = (self.rear + 1) % self.n
        return True

    def deleteFront(self):
        if self.isEmpty():
            return False
        self.front = (self.front + 1) % self.n
        return True

    def deleteLast(self):
        if self.isEmpty():
            return False
        self.rear = (self.rear - 1) % self.n
        return True

    def getFront(self):
        return -1 if self.isEmpty() else self.q[self.front]

    def getRear(self):
        return -1 if self.isEmpty() else self.q[(self.rear - 1) % self.n]


rng = random.Random(4)
for _ in range(200):
    k = rng.choice([1, 2, 3, 5, 9])
    names, args = random_script(rng, k, 50, rng.choice(["insert", "delete", "mixed"]))
    assert run_ops(MyDeque, names, args) == run_ops(RefDeque, names, args)
print("Bài 4 OK")

Bài 4 OK


In [40]:
# Bài 5 (a)
class MyCircularQueue2:
    def __init__(self, k):
        self.n = k + 1
        self.q = [0] * self.n
        self.front = self.rear = 0

    def enQueue(self, v):
        if (self.rear + 1) % self.n == self.front:
            return False
        self.q[self.rear] = v
        self.rear = (self.rear + 1) % self.n
        return True

    def deQueue(self):
        if self.front == self.rear:
            return False
        self.front = (self.front + 1) % self.n
        return True

    def Front(self):
        return -1 if self.front == self.rear else self.q[self.front]

    def Rear(self):
        return -1 if self.front == self.rear else self.q[(self.rear - 1) % self.n]

    def isEmpty(self):
        return self.front == self.rear

    def isFull(self):
        return (self.rear + 1) % self.n == self.front


names = ["MyCircularQueue", "enQueue", "enQueue", "enQueue", "enQueue", "Rear", "isFull", "deQueue", "enQueue", "Rear"]
args = [[3], [1], [2], [3], [4], [], [], [], [4], []]
assert run_ops(MyCircularQueue2, names, args) == [None, True, True, True, False, 3, True, True, True, 4]
rng = random.Random(51)
for _ in range(200):
    k = rng.randint(1, 6)
    nm, ag = ["MyCircularQueue"], [[k]]
    for _ in range(40):
        op = rng.choice(["enQueue", "deQueue", "Front", "Rear", "isEmpty", "isFull"])
        nm.append(op)
        ag.append([rng.randint(0, 9)] if op == "enQueue" else [])
    assert run_ops(MyCircularQueue2, nm, ag) == run_ops(MyCircularQueue, nm, ag)

# Bài 5 (b)
class OverwriteBuffer:
    def __init__(self, k):
        self.q, self.cap, self.front, self.size = [0] * k, k, 0, 0

    def push(self, x):
        if self.size < self.cap:
            self.q[(self.front + self.size) % self.cap] = x
            self.size += 1
        else:                                       # đầy: ghi đè phần tử cũ nhất (ở front)
            self.q[self.front] = x
            self.front = (self.front + 1) % self.cap

    def to_list(self):
        return [self.q[(self.front + i) % self.cap] for i in range(self.size)]


b = OverwriteBuffer(3)
for x in range(1, 6):
    b.push(x)
assert b.to_list() == [3, 4, 5]
rng = random.Random(52)
for _ in range(200):
    k = rng.randint(1, 6)
    buf, ref = OverwriteBuffer(k), deque(maxlen=k)
    for _ in range(30):
        x = rng.randint(0, 99)
        buf.push(x)
        ref.append(x)
        assert buf.to_list() == list(ref)
print("Bài 5 OK")

Bài 5 OK


## 20. Template nhớ nhanh + Checklist trước khi submit

**Template (ring buffer `front` + `size`):**

```python
class MyCircularDeque:

    def __init__(self, k: int):
        self.q = [0] * k
        self.front = 0
        self.size = 0
        self.cap = k

    def insertFront(self, value: int) -> bool:
        if self.size == self.cap:
            return False
        self.front = (self.front - 1) % self.cap       # lùi
        self.q[self.front] = value
        self.size += 1
        return True

    def insertLast(self, value: int) -> bool:
        if self.size == self.cap:
            return False
        self.q[(self.front + self.size) % self.cap] = value
        self.size += 1
        return True

    def deleteFront(self) -> bool:
        if self.size == 0:
            return False
        self.front = (self.front + 1) % self.cap       # tiến
        self.size -= 1
        return True

    def deleteLast(self) -> bool:
        if self.size == 0:
            return False
        self.size -= 1
        return True

    def getFront(self) -> int:
        return -1 if self.size == 0 else self.q[self.front]

    def getRear(self) -> int:
        return -1 if self.size == 0 else self.q[(self.front + self.size - 1) % self.cap]

    def isEmpty(self) -> bool:
        return self.size == 0

    def isFull(self) -> bool:
        return self.size == self.cap
```

**Câu thần chú:** *ô đầu `front`; ô cuối `(front + size - 1) % cap`; chèn đầu thì lùi `front`; xóa đầu thì tiến `front`; xóa cuối chỉ giảm `size`.*

**Checklist trước khi submit:**

- [ ] Có cách phân biệt rỗng/đầy rõ ràng (`size`, hoặc mảng `k + 1` ô)?
- [ ] Mọi phép lùi/tiến chỉ số đều có `% cap`?
- [ ] `getRear` dùng `(front + size - 1) % cap` (không phải `front + size`)?
- [ ] `deleteFront` vừa tiến `front` vừa giảm `size`; `deleteLast` giảm `size`?
- [ ] `getFront`/`getRear` trả `-1` khi rỗng; lệnh chèn/xóa trả `False` khi đầy/rỗng?
- [ ] Không dùng `deque(maxlen=k)` (nó không từ chối khi đầy)?
- [ ] Đã thử: `k = 1`, thao tác khi rỗng, đầy rồi xóa hết rồi đầy lại, quấn vòng nhiều lần, ví dụ của đề?